# Engine Benchmark — pandas vs Spark vs DuckDB (Synthetic Minute Bars)

> **What this notebook does** — a deterministic benchmark of **pull** (read + date filter) and
> **calculation** (per-symbol aggregation) on **synthetic minute bars**, run with **pandas**,
> **PySpark** and **DuckDB**. All engines read the *exact same files* and compute the *exact same
> result*; the only thing that varies is wall-clock time.

## Why deterministic

- **No random tickers.** The ticker list is always the *first N tickers in sorted order* from the
  mounted Kaggle datasets — the same set on every run.
- **Fixed windows.** Same month(s), same full-minute scan, same aggregation — only the engine changes.
- **No S3 / no AWS** — everything is read from `/kaggle/input` (the `dsptlp/synthetic-market-data`
  dataset mount), so the benchmark costs nothing and is fully reproducible.

## The four test groups

| Group | Window | Universe | What it shows |
|---|---|---|---|
| **A — baseline** | 1 month (June 2024) | 300 tickers | head-to-head on a small scan |
| **B — wider window** | 12 months (Jun 2024–May 2025) | 300 tickers | how engines scale when the *filtered* set grows ~12× |
| **C — full minute data** | no date filter (all bars) | 300 tickers | raw full-scan throughput on the minute dataset |
| **D — bigger universe** | 1 month (June 2024) | 2000 tickers | how engines scale when the *file set* grows ~7× (pandas skipped: RAM-bound above ~800 tickers in a 16 GB session) |
| **E — full frame x full universe** | full range (Feb 2024–Jan 2026) | 10,000 tickers | the minutes-scale test: ~40 GB scan per engine (DuckDB + Spark only) |
| **F — EVERYTHING** | full range (Feb 2024–Jan 2026) | ALL 20,000 tickers | the max: ~80 GB scan per engine (DuckDB + Spark only) |
| **G — operations battery** | full range (Feb 2024–Jan 2026) | 600 tickers | same data, 7 different operations (count / sum / avg / stats / distinct / filtered / window) — how each engine handles different query shapes |
| **H — stress test** | full range (Feb 2024–Jan 2026) | 500 → 20,000 tickers | push each engine in a RAM-capped subprocess until it DIES — who can handle the most data, and where each breaks |

Groups A–D finish in seconds; **E/F are the minutes-scale scans**; **G compares query shapes**;
**H finds each engine's breaking point** (a dead child never kills the notebook).


In [ ]:
# ============================================================================
# Setup -- engines, dataset discovery, deterministic inputs
# ============================================================================

!pip install -q duckdb --upgrade

import os
import sys
import glob as _glob
import time

import numpy as np
import pandas as pd
import duckdb

import plotly.express as px

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

# --- Spark (local mode inside the Kaggle session) ---------------------------
# 8 GB driver heap so toPandas() can materialize large results without OOM.
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("engine-benchmark")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# --- Locate the synthetic minute-bar files (mount path varies) --------------
minute_dirs = sorted(_glob.glob("/kaggle/input/**/minute_data_final", recursive=True))
if not minute_dirs:
    raise SystemExit("synthetic minute_data_final not found in /kaggle/input - attach "
                     "dsptlp/synthetic-market-data and dsptlp/synthetic-market-data-02")

ALL_FILES = sorted(
    os.path.join(d, f)
    for d in minute_dirs
    for f in os.listdir(d)
    if f.endswith(".parquet")
)
print(f"Found {len(ALL_FILES):,} synthetic ticker files")

# --- Deterministic inputs ----------------------------------------------------
# First N tickers in sorted order (deterministic; note the mount path sort picks
# shard-2 tickers first, e.g. SGT11230.. -- still the same set every run).
N_TICKERS = 300            # groups A/B/C universe
N_BIG     = 2000           # group D universe
N_OPS     = 600            # group G ops-battery universe (600 files full-frame is the safe ceiling; pandas skipped)
N_MAX     = 10000          # group E universe (~40 GB scan; the minutes-scale test)
PANDAS_MAX = 800           # pandas is RAM-bound above this many files in a 16 GB session

N_RUNS    = 3              # repeats per task for groups A; 2 for the heavier groups

# --- Group G operation definitions (all run on the same files/window) --------
OPS_FILTERS = {
    "filtered": {"min_volume": 100_000, "min_close": 50.0},   # WHERE volume > X AND close > Y
}
WINDOW_BARS = 20           # rolling-average window size for the "window" op

# --- Windows (epoch-ms, matching the data's `date` column) -------------------
W1_LO  = int(pd.Timestamp("2024-06-01", tz="UTC").timestamp() * 1000)   # June 2024
W1_HI  = int(pd.Timestamp("2024-07-01", tz="UTC").timestamp() * 1000)
W12_LO = int(pd.Timestamp("2024-06-01", tz="UTC").timestamp() * 1000)   # Jun 2024 - May 2025
W12_HI = int(pd.Timestamp("2025-06-01", tz="UTC").timestamp() * 1000)
WALL_LO = int(pd.Timestamp("2024-02-01", tz="UTC").timestamp() * 1000)  # full data frame
WALL_HI = int(pd.Timestamp("2026-01-01", tz="UTC").timestamp() * 1000)
MAX_HI = int(pd.Timestamp("2027-01-01", tz="UTC").timestamp() * 1000)   # full scan ceiling

# Explicit schema -- prices/volume double, symbol/trades long (matches the synthetic generator).
MINUTE_SCHEMA = StructType([
    StructField("symbol", StringType(), True),
    StructField("date",   LongType(),   True),
    StructField("open",   DoubleType(), True),
    StructField("high",   DoubleType(), True),
    StructField("low",    DoubleType(), True),
    StructField("close",  DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("vwap",   DoubleType(), True),
    StructField("trades", LongType(),   True),
])

# --- DuckDB connection (persistent across runs) -----------------------------
con = duckdb.connect()

print(f"Groups A/B/C universe : {N_TICKERS} files = {ALL_FILES[0].split('/')[-1]} .. {ALL_FILES[N_TICKERS-1].split('/')[-1]}")
print(f"Group  D     universe : {N_BIG} files = {ALL_FILES[0].split('/')[-1]} .. {ALL_FILES[N_BIG-1].split('/')[-1]}")
print(f"Group  E     universe : {N_MAX} files = {ALL_FILES[0].split('/')[-1]} .. {ALL_FILES[N_MAX-1].split('/')[-1]} (~40 GB full-frame scan)")
print(f"Group  F     universe : ALL {len(ALL_FILES)} files (~80 GB full-frame scan)")
print(f"Group  G     ops      : {N_BIG} files x full frame, {WINDOW_BARS}-bar rolling window")
print("Windows: A=1mo D=1mo  B=12mo  C/E/F/G=full-frame")
print("Engines ready")


In [ ]:
# ============================================================================
# Benchmark helper -- best-of-N wall time
# ============================================================================

def bench(fn, runs=N_RUNS, warmup=True):
    if warmup:
        fn()                        # warm OS page cache once (benefits all engines equally)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        result = fn()
        times.append(time.perf_counter() - t0)
    return min(times), times, result


def fmt(seconds):
    return f"{seconds:8.2f}s" if seconds >= 60 else f"{seconds * 1000:8.1f} ms"


def restart_spark():
    # Stop and recreate the Spark session, dropping the JVM's heap.
    # Needed between the huge groups so a bloated executor doesn't OOM later ops.
    global spark
    try:
        spark.stop()
    except Exception:
        pass
    import time as _t
    _t.sleep(2)
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("engine-benchmark")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .config("spark.driver.memory", "6g")
        .config("spark.driver.maxResultSize", "0")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session restarted (fresh JVM)")


# ============================================================================
# Engine functions (pull = scan + filter -> materialized ROW COUNT;
#                   calc = scan + filter + per-symbol aggregation -> DataFrame)
# ============================================================================

def pull_pandas(files, lo, hi):
    df = pd.read_parquet(files)                                    # read ALL columns
    return int(((df["date"] >= lo) & (df["date"] < hi)).sum())     # materialized row count

def pull_duckdb(files, lo, hi):
    return int(con.execute(f"SELECT COUNT(*) FROM read_parquet({files!r}) "
                           f"WHERE date >= {lo} AND date < {hi}").fetchone()[0])

def pull_spark(files, lo, hi):
    df = (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
          .filter(F.col("date") >= lo)
          .filter(F.col("date") <  hi))
    return int(df.count())                                         # forces the full scan

def calc_pandas(files, lo, hi):
    df = pd.read_parquet(files)
    m = df[(df["date"] >= lo) & (df["date"] < hi)]
    agg = (m.groupby("symbol")
             .agg(volume=("volume", "sum"), vwap=("vwap", "mean"),
                  low=("low", "min"), high=("high", "max"),
                  bars=("date", "count")))
    return agg.reset_index().sort_values("symbol")

def calc_duckdb(files, lo, hi):
    return con.execute(f"SELECT symbol, SUM(volume) AS volume, AVG(vwap) AS vwap, "
                       f"MIN(low) AS low, MAX(high) AS high, COUNT(*) AS bars "
                       f"FROM read_parquet({files!r}) "
                       f"WHERE date >= {lo} AND date < {hi} "
                       f"GROUP BY symbol ORDER BY symbol").df()

def calc_spark(files, lo, hi):
    df = (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
          .filter(F.col("date") >= lo)
          .filter(F.col("date") <  hi))
    agg = (df.groupBy("symbol")
             .agg(F.sum("volume").alias("volume"),
                  F.avg("vwap").alias("vwap"),
                  F.min("low").alias("low"),
                  F.max("high").alias("high"),
                  F.count("*").alias("bars")))
    return agg.orderBy("symbol").toPandas()

PULLERS = {"pandas": pull_pandas, "duckdb": pull_duckdb, "spark": pull_spark}
CALCERS = {"pandas": calc_pandas, "duckdb": calc_duckdb, "spark": calc_spark}


# ============================================================================
# Task runner -- runs pull + calc for a set of engines, prints a mini table
# ============================================================================

def run_group(tag, label, files, lo, hi, engines, runs):
    print(f"\n========== Group {tag} : {label} ==========")

    pulls, calcs = {}, {}
    for e in engines:
        pulls[e] = bench(lambda e=e: PULLERS[e](files, lo, hi), runs=runs)
        calcs[e] = bench(lambda e=e: CALCERS[e](files, lo, hi), runs=runs)

    print(f"  {'engine':<8} {'PULL best':>10} {'PULL median':>12}   rows")
    for e in engines:
        best, times, n = pulls[e]
        print(f"  {e:<8} {fmt(best):>10} {fmt(float(np.median(times))):>12}   {n:>12,}")

    print(f"  {'engine':<8} {'CALC best':>10} {'CALC median':>12}   agg rows")
    for e in engines:
        best, times, agg = calcs[e]
        print(f"  {e:<8} {fmt(best):>10} {fmt(float(np.median(times))):>12}   {len(agg):>12,}")

    return {"pull": pulls, "calc": calcs}


## Run the groups

- **Group A — baseline**: 300 tickers x 1 month (June 2024). All three engines.
- **Group B — wider window**: 300 tickers x 12 months (Jun 2024–May 2025). All three engines.
- **Group C — full minute data**: 300 tickers x ALL bars (no date filter). All three engines.
- **Group D — bigger universe**: 2000 tickers x 1 month (June 2024). DuckDB + Spark only —
  pandas would need ~20 GB+ RAM for the 2000-file full-file read (safe cap ≈ `PANDAS_MAX`).
- **Group E — full frame x 10,000 tickers**: ~40 GB scan per engine (~2.7 B rows). Minutes.
- **Group F — full frame x ALL 20,000 tickers**: the max — ~80 GB scan per engine. Minutes.
- **Group G — operations battery**: 600 tickers x full frame, seven different query shapes —
  `count`, `sum`, `avg`, `stats` (SUM/AVG/MIN/MAX/STDDEV), `distinct` (trading days),
  `filtered` (WHERE volume > 100k AND close > 50), `window` (20-bar rolling mean).
- **Group H — stress test**: runs each engine in a RAM-capped subprocess on ever-larger file
  sets (500 → 20,000) until the engine DIES. Shows the limits: which engine can handle the
  most data, and where each one breaks (a crashed child never kills the notebook).


In [ ]:
# ============================================================================
# Groups A, B, C -- 300 tickers; window widens, then full minute scan
# ============================================================================

FILES_300 = ALL_FILES[:N_TICKERS]

print(">>> Group A: baseline (1 month, 300 tickers)")
results_A = run_group("A", "1 month x 300 tickers",
                      FILES_300, W1_LO, W1_HI,
                      engines=("pandas", "duckdb", "spark"), runs=3)

print(">>> Group B: 12-month window (300 tickers)")
results_B = run_group("B", "12 months x 300 tickers",
                      FILES_300, W12_LO, W12_HI,
                      engines=("pandas", "duckdb", "spark"), runs=2)

print(">>> Group C: full minute data, no date filter (300 tickers)")
results_C = run_group("C", "full minute data x 300 tickers",
                      FILES_300, WALL_LO, WALL_HI,
                      engines=("pandas", "duckdb", "spark"), runs=2)


In [ ]:
# ============================================================================
# Group D -- bigger universe: 2000 tickers x 1 month (DuckDB + Spark)
# ============================================================================

FILES_BIG = ALL_FILES[:N_BIG]
print(f">>> Group D: 1 month x {N_BIG} tickers (pandas skipped: RAM-bound "
      f"above ~{PANDAS_MAX} files in a 16 GB session)")
results_D = run_group("D", f"1 month x {N_BIG} tickers",
                      FILES_BIG, W1_LO, W1_HI,
                      engines=("duckdb", "spark"), runs=2)


In [ ]:
# ============================================================================
# Group E -- 10,000 tickers x full 2-year frame (~40 GB scan, ~2.7 B rows)
# ============================================================================

FILES_MAX = ALL_FILES[:N_MAX]
print(f">>> Group E: full frame x {N_MAX} tickers (~40 GB scan; minutes)")
results_E = run_group("E", f"full frame x {N_MAX} tickers",
                      FILES_MAX, WALL_LO, WALL_HI,
                      engines=("duckdb", "spark"), runs=1)


In [ ]:
# ============================================================================
# Group F -- the MAX: ALL 20,000 tickers x full 2-year frame
# (~80 GB scan per engine, ~3.9 B rows). Expect MANY minutes.
# ============================================================================

FILES_ALL = ALL_FILES[:]  # every ticker in the mounted datasets
print(f">>> Group F: full frame x ALL {len(FILES_ALL)} tickers (~80 GB scan; "
      f"this one takes many minutes)")
results_F = run_group("F", f"full frame x {len(FILES_ALL)} tickers",
                      FILES_ALL, WALL_LO, WALL_HI,
                      engines=("duckdb", "spark"), runs=1)

# Free the JVM heap that Group F (80 GB scan) bloated before running the ops battery.
restart_spark()


In [ ]:
# ============================================================================
# Group G -- operations battery: same 2000-ticker full-frame data, 7 query shapes
# Each op returns a per-symbol aggregation; engines must agree exactly.
# ============================================================================

FILES_OPS = ALL_FILES[:N_OPS]
OPS_LO, OPS_HI = WALL_LO, WALL_HI
VOL_MIN, CLOSE_MIN = OPS_FILTERS["filtered"]["min_volume"], OPS_FILTERS["filtered"]["min_close"]
WB = WINDOW_BARS


def op_count(files, lo, hi):
    out = {}
    out["pandas"] = lambda: (pd.read_parquet(files)
                             .pipe(lambda d: d[(d["date"] >= lo) & (d["date"] < hi)])
                             .groupby("symbol").size().reset_index(name="n").sort_values("symbol"))
    out["duckdb"] = lambda: con.execute(f"SELECT symbol, COUNT(*) AS n FROM read_parquet({files!r}) "
                                        f"WHERE date >= {lo} AND date < {hi} "
                                        f"GROUP BY symbol ORDER BY symbol").df()
    out["spark"] = lambda: (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
                            .filter(F.col("date") >= lo).filter(F.col("date") < hi)
                            .groupBy("symbol").count().orderBy("symbol").toPandas())
    return out


def op_sum(files, lo, hi):
    out = {}
    out["pandas"] = lambda: (pd.read_parquet(files)
                             .pipe(lambda d: d[(d["date"] >= lo) & (d["date"] < hi)])
                             .groupby("symbol")["volume"].sum().reset_index(name="v").sort_values("symbol"))
    out["duckdb"] = lambda: con.execute(f"SELECT symbol, SUM(volume) AS v FROM read_parquet({files!r}) "
                                        f"WHERE date >= {lo} AND date < {hi} "
                                        f"GROUP BY symbol ORDER BY symbol").df()
    out["spark"] = lambda: (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
                            .filter(F.col("date") >= lo).filter(F.col("date") < hi)
                            .groupBy("symbol").agg(F.sum("volume").alias("v")).orderBy("symbol").toPandas())
    return out


def op_avg(files, lo, hi):
    out = {}
    out["pandas"] = lambda: (pd.read_parquet(files)
                             .pipe(lambda d: d[(d["date"] >= lo) & (d["date"] < hi)])
                             .groupby("symbol")["vwap"].mean().reset_index(name="v").sort_values("symbol"))
    out["duckdb"] = lambda: con.execute(f"SELECT symbol, AVG(vwap) AS v FROM read_parquet({files!r}) "
                                        f"WHERE date >= {lo} AND date < {hi} "
                                        f"GROUP BY symbol ORDER BY symbol").df()
    out["spark"] = lambda: (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
                            .filter(F.col("date") >= lo).filter(F.col("date") < hi)
                            .groupBy("symbol").agg(F.avg("vwap").alias("v")).orderBy("symbol").toPandas())
    return out


def op_stats(files, lo, hi):
    out = {}
    out["pandas"] = lambda: (pd.read_parquet(files)
                             .pipe(lambda d: d[(d["date"] >= lo) & (d["date"] < hi)])
                             .groupby("symbol").agg(vol=("volume", "sum"), vwap=("vwap", "mean"),
                                                    lo=("low", "min"), hi=("high", "max"),
                                                    sd=("close", "std"), n=("date", "count"))
                             .reset_index().sort_values("symbol"))
    out["duckdb"] = lambda: con.execute(f"SELECT symbol, SUM(volume) AS vol, AVG(vwap) AS vwap, "
                                        f"MIN(low) AS lo, MAX(high) AS hi, STDDEV(close) AS sd, "
                                        f"COUNT(*) AS n FROM read_parquet({files!r}) "
                                        f"WHERE date >= {lo} AND date < {hi} "
                                        f"GROUP BY symbol ORDER BY symbol").df()
    out["spark"] = lambda: (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
                            .filter(F.col("date") >= lo).filter(F.col("date") < hi)
                            .groupBy("symbol")
                            .agg(F.sum("volume").alias("vol"), F.avg("vwap").alias("vwap"),
                                 F.min("low").alias("lo"), F.max("high").alias("hi"),
                                 F.stddev("close").alias("sd"), F.count("*").alias("n"))
                            .orderBy("symbol").toPandas())
    return out


def op_distinct(files, lo, hi):
    out = {}
    out["pandas"] = lambda: (pd.read_parquet(files)
                             .pipe(lambda d: d[(d["date"] >= lo) & (d["date"] < hi)])
                             .groupby("symbol")["date"].nunique().reset_index(name="n").sort_values("symbol"))
    out["duckdb"] = lambda: con.execute(f"SELECT symbol, COUNT(DISTINCT date) AS n FROM read_parquet({files!r}) "
                                        f"WHERE date >= {lo} AND date < {hi} "
                                        f"GROUP BY symbol ORDER BY symbol").df()
    out["spark"] = lambda: (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
                            .filter(F.col("date") >= lo).filter(F.col("date") < hi)
                            .groupBy("symbol").agg(F.countDistinct("date").alias("n"))
                            .orderBy("symbol").toPandas())
    return out


def op_filtered(files, lo, hi):
    out = {}
    out["pandas"] = lambda: (pd.read_parquet(files)
                             .pipe(lambda d: d[(d["date"] >= lo) & (d["date"] < hi)])
                             .pipe(lambda d: d[(d["volume"] > VOL_MIN) & (d["close"] > CLOSE_MIN)])
                             .groupby("symbol")["volume"].sum().reset_index(name="v").sort_values("symbol"))
    out["duckdb"] = lambda: con.execute(f"SELECT symbol, SUM(volume) AS v FROM read_parquet({files!r}) "
                                        f"WHERE date >= {lo} AND date < {hi} "
                                        f"AND volume > {VOL_MIN} AND close > {CLOSE_MIN} "
                                        f"GROUP BY symbol ORDER BY symbol").df()
    out["spark"] = lambda: (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
                            .filter(F.col("date") >= lo).filter(F.col("date") < hi)
                            .filter(F.col("volume") > VOL_MIN).filter(F.col("close") > CLOSE_MIN)
                            .groupBy("symbol").agg(F.sum("volume").alias("v")).orderBy("symbol").toPandas())
    return out


def op_window(files, lo, hi):
    # 20-bar rolling mean of close per symbol -> per-symbol COUNT of windowed bars.
    # (We return a small per-symbol frame, NOT the full per-bar frame -- materializing
    # every bar for 600 tickers x full frame would OOM the driver.)
    out = {}
    out["pandas"] = lambda: (pd.read_parquet(files)
                             .pipe(lambda d: d[(d["date"] >= lo) & (d["date"] < hi)])
                             .sort_values(["symbol", "date"])
                             .assign(r=lambda m: m.groupby("symbol")["close"]
                                     .rolling(WB, min_periods=1).mean()
                                     .reset_index(level=0, drop=True).values)
                             .groupby("symbol").size().reset_index(name="n").sort_values("symbol"))
    out["duckdb"] = lambda: con.execute(f"SELECT symbol, COUNT(*) AS n FROM ("
                                        f"SELECT symbol, AVG(close) OVER (PARTITION BY symbol "
                                        f"ORDER BY date ROWS BETWEEN {WB-1} PRECEDING AND CURRENT ROW) AS r "
                                        f"FROM read_parquet({files!r}) "
                                        f"WHERE date >= {lo} AND date < {hi}) "
                                        f"GROUP BY symbol ORDER BY symbol").df()
    out["spark"] = lambda: (spark.read.schema(MINUTE_SCHEMA).parquet(*files)
                            .filter(F.col("date") >= lo).filter(F.col("date") < hi)
                            .withColumn("r", F.avg("close").over(
                                Window.partitionBy("symbol").orderBy("date")
                                      .rowsBetween(-(WB-1), 0)))
                            .groupBy("symbol").count().orderBy("symbol").toPandas())
    return out


OPS = {"count": op_count, "sum": op_sum, "avg": op_avg,
       "stats": op_stats, "distinct": op_distinct,
       "filtered": op_filtered, "window": op_window}

print(f">>> Group G: operations battery x {N_OPS} tickers, full frame "
      f"(duckdb + spark only -- pandas is RAM-bound at this scale)")
results_G = {}
for opname, opfn in OPS.items():
    impls = opfn(FILES_OPS, OPS_LO, OPS_HI)
    results_G[opname] = {}
    print(f"  --- op: {opname} ---")
    for engine in ("duckdb", "spark"):
        results_G[opname][engine] = bench(impls[engine], runs=1, warmup=False)
        best, times, _ = results_G[opname][engine]
        print(f"  {engine:<8} {fmt(best):>10}")


In [ ]:
# ============================================================================
# Group H -- STRESS TEST: push each engine until it dies.
# Each engine runs in a SUBPROCESS with a RAM cap, so an OOM-killed child
# NEVER takes down the notebook. We record the largest dataset each engine
# can handle (who survives the most data) and where each engine dies.
# ============================================================================

import subprocess
import sys as _sys
import psutil

# Free the main notebook's Spark JVM before spawning stress-test children
# (each child starts its own Spark; keeping the parent's 6 GB heap alive
# would eat the whole 16 GB session).
try:
    spark.stop()
    print("Stopped main Spark session before stress test")
except Exception:
    pass
import gc as _gc
_gc.collect()   # release the JVM's python-side buffers before the stress children

STRESS_SIZES  = [100, 200, 500, 1000, 2000, 4000, 8000, 16000, 20000]   # files (increasing data)
MEM_BUDGET_GB = 6.0           # kill the child if its RSS exceeds this -- must stay well under
                              # the 16 GB session (child + parent must never OOM the box)
STRESS_TIMEOUT_S = 2400       # per-run wall-clock cap (40 min)

# The child is fully self-contained (it re-globs /kaggle/input itself).
STRESS_CHILD = r"""
import sys, os, glob, time
engine = sys.argv[1]
n = int(sys.argv[2])
dirs = sorted(glob.glob("/kaggle/input/**/minute_data_final", recursive=True))
files = sorted(os.path.join(d, f) for d in dirs for f in os.listdir(d)
               if f.endswith(".parquet"))[:n]
import pandas as pd
import duckdb
t0 = time.time()
if engine == "pandas":
    df = pd.read_parquet(files)                     # loads EVERYTHING into RAM
    agg = df.groupby("symbol").agg(volume=("volume", "sum"), bars=("date", "count"))
    print(f"OK n={n} symbols={len(agg)} seconds={time.time()-t0:.1f}")
elif engine == "duckdb":
    con = duckdb.connect()
    nrows = con.execute(f"SELECT COUNT(*) FROM read_parquet({files!r})").fetchone()[0]
    nsym  = con.execute(f"SELECT COUNT(DISTINCT symbol) FROM read_parquet({files!r})").fetchone()[0]
    print(f"OK n={n} rows={nrows:,} symbols={nsym} seconds={time.time()-t0:.1f}")
elif engine == "spark":
    from pyspark.sql import SparkSession, Window
    from pyspark.sql import functions as F
    from pyspark.sql.types import *
    SCHEMA = StructType([
        StructField("symbol", StringType(), True), StructField("date", LongType(), True),
        StructField("open", DoubleType(), True), StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True), StructField("close", DoubleType(), True),
        StructField("volume", DoubleType(), True), StructField("vwap", DoubleType(), True),
        StructField("trades", LongType(), True)])
    spark = SparkSession.builder.master("local[*]").appName("stress").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    df = spark.read.schema(SCHEMA).parquet(*files)
    nrows = df.count()
    nsym  = df.select("symbol").distinct().count()
    print(f"OK n={n} rows={nrows:,} symbols={nsym} seconds={time.time()-t0:.1f}")
else:
    print("BADENGINE"); sys.exit(2)
"""


def run_stress(engine, n_files):
    """Run one engine on n_files in a capped subprocess; return a result row."""
    import threading
    cmd = [_sys.executable, "-c", STRESS_CHILD, engine, str(n_files)]
    t0 = time.time()
    # Drain stdout in a thread so a verbose child (spark logs) can never block
    # on a full pipe buffer; keep only the tail for the status message.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, start_new_session=True)
    chunks = []
    def _drain():
        for line in proc.stdout:
            chunks.append(line)
            if len(chunks) > 400:
                chunks.pop(0)
    th = threading.Thread(target=_drain, daemon=True)
    th.start()
    peak = 0
    status = "OK"
    try:
        p = psutil.Process(proc.pid)
    except Exception:
        p = None
    while proc.poll() is None:
        if p is not None:
            try:
                kids = p.children(recursive=True)
                rss = p.memory_info().rss + sum(k.memory_info().rss for k in kids)
                peak = max(peak, rss)
            except Exception:
                pass
        if peak > MEM_BUDGET_GB * 1024 ** 3:
            proc.kill()
            status = f"DIED_RAM>{MEM_BUDGET_GB:.0f}GB"
            break
        if time.time() - t0 > STRESS_TIMEOUT_S:
            proc.kill()
            status = "TIMEOUT"
            break
        time.sleep(0.05)
    th.join(timeout=5)
    proc.wait()
    seconds = round(time.time() - t0, 1)
    out = "".join(chunks)
    # Success = our "OK n=..." marker anywhere in the child's stdout (engines
    # like spark/duckdb print logging noise BEFORE the marker).
    ok = (status == "OK" and proc.returncode == 0 and "OK n=" in out)
    if not ok and status == "OK":
        status = f"DIED_rc={proc.returncode}"
    return {"engine": engine, "n_files": n_files, "status": status,
            "seconds": seconds, "peak_gb": round(peak / 1024 ** 3, 2),
            "out": out.strip()[-120:]}


print(f">>> Group H: stress test -- increasing data until each engine dies "
      f"(RAM cap {MEM_BUDGET_GB} GB per child)")
stress_rows = []
for engine in ("pandas", "duckdb", "spark"):
    for n in STRESS_SIZES:
        row = run_stress(engine, n)
        stress_rows.append(row)
        print(f"  {engine:<7} n={n:>6,} -> {row['status']:<18} "
              f"{row['seconds']:>7.1f}s peak={row['peak_gb']:>5.1f}GB")
        if row["status"] != "OK":
            print(f"    !! {engine} DIED at {n:,} files -- engine limit reached")
            break

stress_df = pd.DataFrame(stress_rows)
print("\nLargest dataset each engine survived:")
for engine in ("pandas", "duckdb", "spark"):
    sub = stress_df[stress_df["engine"] == engine]
    ok = sub[sub["status"] == "OK"]
    died = sub[sub["status"] != "OK"]
    max_ok = ok["n_files"].max() if len(ok) else 0
    die_at = died["n_files"].min() if len(died) else None
    est_gb = max_ok * 4.0 / 1000
    print(f"  {engine:<8} survived {max_ok:>6,} files (~{est_gb:.0f} GB full-frame)"
          + (f"  | died at {die_at:,} files" if die_at else "  | survived everything"))


In [ ]:
# ============================================================================
# Correctness -- every engine in every group MUST return identical results
# ============================================================================

def check(a, b, tag):
    a = a.sort_values("symbol").reset_index(drop=True)
    b = b.sort_values("symbol").reset_index(drop=True)
    assert list(a["symbol"]) == list(b["symbol"]), f"{tag}: ticker order mismatch"
    for col in ("volume", "vwap", "low", "high", "bars"):
        err = float((a[col].astype(float) - b[col].astype(float)).abs().max())
        assert err < 1e-3, f"{tag}: column {col} differs by {err}"
    return True


def verify_group(tag, results):
    engines = list(results["pull"].keys())
    rows = {e: results["pull"][e][2] for e in engines}
    aggs = {e: results["calc"][e][2] for e in engines}
    assert len(set(rows.values())) == 1, f"{tag}: row counts differ {rows}"
    base = engines[0]
    for e in engines[1:]:
        check(aggs[base], aggs[e], f"{tag} {base} vs {e}")
    print(f"OK Group {tag}: {'=='.join(engines)} | rows pulled {list(rows.values())[0]:,} "
          f"| symbols {len(aggs[base]):,} (exact tickers, exact aggs)")


for tag, results in (("A", results_A), ("B", results_B),
                     ("C", results_C), ("D", results_D), ("E", results_E),
                     ("F", results_F)):
    verify_group(tag, results)


# --- Group G correctness: every op must match across engines ---------------
def check_cols(a, b, tag):
    # column-agnostic comparison: shared columns must match (ops return different
    # schemas: n / v / vol,vwap,lo,hi,sd / etc.)
    a = a.sort_values("symbol").reset_index(drop=True)
    b = b.sort_values("symbol").reset_index(drop=True)
    assert list(a["symbol"]) == list(b["symbol"]), f"{tag}: ticker order mismatch"
    for col in set(a.columns) & set(b.columns):
        if col in ("symbol", "date"):
            continue
        err = float((a[col].astype(float) - b[col].astype(float)).abs().max())
        assert err < 1e-3, f"{tag}: column {col} differs by {err}"
    return True


def verify_op(opname, results):
    engines = list(results.keys())
    base = engines[0]
    res_base = results[base][2]
    for e in engines[1:]:
        res_e = results[e][2]
        if isinstance(res_base, pd.DataFrame):
            check_cols(res_base, res_e, f"G-{opname} {base} vs {e}")
        else:
            assert res_base == res_e, f"G-{opname}: {base}={res_base} vs {e}={res_e}"
    print(f"OK G-{opname}: {'=='.join(engines)}")


for opname, results in results_G.items():
    verify_op(opname, results)


# --- Group H soft sanity: duckdb should survive everything, pandas dies first -
try:
    _hmax = {e: max(r["n_files"] for r in stress_rows if r["engine"] == e and r["status"] == "OK")
             for e in ("pandas", "duckdb", "spark")}
    print(f"Group H capacity (files survived): pandas={_hmax['pandas']:,} duckdb={_hmax['duckdb']:,} "
          f"spark={_hmax['spark']:,}")
except Exception as e:
    print("Group H sanity skipped:", e)


In [ ]:
# ============================================================================
# RESULTS SUMMARY -- combined table + a full dashboard figure + detail CSVs
# ============================================================================

# --- Gather every result row ------------------------------------------------
rows = []
for tag, results in (("A", results_A), ("B", results_B),
                     ("C", results_C), ("D", results_D), ("E", results_E),
                     ("F", results_F)):
    for kind in ("pull", "calc"):
        for engine, (best, times, _) in results[kind].items():
            rows.append({
                "group": tag, "task": kind, "engine": engine,
                "best_s": round(best, 4),
                "median_s": round(float(np.median(times)), 4),
            })
summary = pd.DataFrame(rows)
summary["label"] = summary["group"] + " " + summary["task"]
summary["best_ms"] = (summary["best_s"] * 1000).round(1)

op_rows = []
for opname, results in results_G.items():
    for engine, (best, times, _) in results.items():
        op_rows.append({"op": opname, "engine": engine,
                        "best_s": round(best, 4),
                        "median_s": round(float(np.median(times)), 4)})
op_summary = pd.DataFrame(op_rows)

stress_df = pd.DataFrame(stress_rows)

# --- Text tables -------------------------------------------------------------
pivot = summary.pivot_table(index="engine", columns="label", values="best_s")
op_pivot = op_summary.pivot_table(index="engine", columns="op", values="best_s")
stress_pivot = stress_df.pivot_table(index="engine", columns="n_files",
                                     values="status", aggfunc="first")

print("=== GROUPS A-F: best wall time (s) ===")
print(pivot.round(2).to_string())
print("\n=== GROUP G: ops battery, best (s) ===")
print(op_pivot.round(2).to_string())
print("\n=== GROUP H: stress (status per file count) ===")
print(stress_pivot.to_string())
print("\n=== GROUP H: largest dataset survived ===")
for engine in ("pandas", "duckdb", "spark"):
    sub = stress_df[stress_df["engine"] == engine]
    ok = sub[sub["status"] == "OK"]
    died = sub[sub["status"] != "OK"]
    max_ok = ok["n_files"].max() if len(ok) else 0
    die_at = died["n_files"].min() if len(died) else None
    print(f"  {engine:<8} survived {max_ok:>6,} files (~{max_ok * 4 / 1000:.0f} GB full-frame)"
          + (f"  | died at {die_at:,} files" if die_at else "  | survived everything"))

# --- Dashboard: one 2x2 figure with everything ------------------------------
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Groups A-F: best wall time (log s)",
                    "Group G: ops battery (log s)",
                    "Group H: stress - data survived (GB)",
                    "Group H: time vs data size (log-log)"),
    horizontal_spacing=0.13, vertical_spacing=0.16)

# (1,1) groups bar
for engine in ("pandas", "duckdb", "spark"):
    sub = summary[summary["engine"] == engine]
    fig.add_trace(go.Bar(x=sub["label"], y=sub["best_s"], name=engine,
                         legendgroup=engine), row=1, col=1)
fig.update_yaxes(type="log", title="seconds (log)", row=1, col=1)

# (1,2) ops bar
for engine in ("pandas", "duckdb", "spark"):
    sub = op_summary[op_summary["engine"] == engine]
    if len(sub):
        fig.add_trace(go.Bar(x=sub["op"], y=sub["best_s"], name=engine,
                             legendgroup=engine, showlegend=False), row=1, col=2)
fig.update_yaxes(type="log", title="seconds (log)", row=1, col=2)

# (2,1) stress survived GB
engines = ("pandas", "duckdb", "spark")
survived_gb = []
for engine in engines:
    sub = stress_df[(stress_df["engine"] == engine) & (stress_df["status"] == "OK")]
    survived_gb.append(sub["n_files"].max() * 4.0 / 1000 if len(sub) else 0)
fig.add_trace(go.Bar(x=engines, y=survived_gb,
                     name="survived data (GB)", marker_color=["#d62728", "#2ca02c", "#1f77b4"],
                     showlegend=False), row=2, col=1)
fig.update_yaxes(title="GB full-frame survived", row=2, col=1)

# (2,2) time vs size for surviving points, log-log
for engine in engines:
    sub = stress_df[(stress_df["engine"] == engine) & (stress_df["status"] == "OK")]
    if len(sub):
        fig.add_trace(go.Scatter(x=sub["n_files"], y=sub["seconds"], mode="lines+markers",
                                 name=engine + " (time)", legendgroup=engine,
                                 line=dict(width=2)), row=2, col=2)
fig.update_xaxes(type="log", title="files (log)", row=2, col=2)
fig.update_yaxes(type="log", title="seconds (log)", row=2, col=2)

fig.update_layout(height=850, title=("Engine benchmark: pandas vs Spark vs DuckDB -- "
                                     "full summary (synthetic data, deterministic)"),
                  barmode="group", legend=dict(orientation="h", y=1.02))
fig.show()

# --- Detail CSVs -------------------------------------------------------------
summary.to_csv("/kaggle/working/benchmark_results.csv", index=False)
op_summary.to_csv("/kaggle/working/benchmark_ops.csv", index=False)
stress_df.to_csv("/kaggle/working/benchmark_stress.csv", index=False)
summary.assign(test="group").to_csv("/kaggle/working/benchmark_all.csv", index=False)
op_summary.assign(test="ops").to_csv("/kaggle/working/benchmark_all.csv", mode="a",
                                     header=False, index=False)
stress_df.assign(test="stress").to_csv("/kaggle/working/benchmark_all.csv", mode="a",
                                       header=False, index=False)
print("Wrote benchmark_results.csv / benchmark_ops.csv / benchmark_stress.csv / benchmark_all.csv")


In [ ]:
# ============================================================================
# Optional: push the scale even further (uncomment to try)
# ============================================================================
# Group F already scans the FULL 20,000-ticker universe (~80 GB). To make the
# ops battery heavier, raise N_BIG before running Group G, or add runs:
# results_G2 = {op: bench(fn, runs=3) for op, fn in ...}
